# 🚀 THE NEW ERA — 1-Click Android APK/AAB & XAPK Merger + XTTS v2 Voice Factory
هذا الدفتر الشامل يجمع **جميع ملفات مشروعك السابقة** مع الملفين المرفوعين حديثاً (`42818f08.html` / `FadeL_Identity_Transfer_Prompt_Studio_VAULTS_FIXED.html` و `com.rmot.aifaceswap_8.0.xapk`):
1. **دمج وتحويل حزمة `com.rmot.aifaceswap_8.0.xapk`** (`com.rmot.aifaceswap.apk` + `config.arm64_v8a.apk`) إلى ملف `.apk` موحد وموقّع جاهز للتثبيت المباشر.
2. **بناء وتوقيع تطبيق أندرويد الكامل `The_New_Era_v1.0_Signed.apk`** الذي يضم جميع الـ 22 استوديو (بما فيها استوديو نقل الهوية، Remake AI Face Swap v8.0، استوديو الكوميكس، المونتير، واستنساخ الصوت) مع التوقيع بمفتاحك `newera-signing-key.p12`.
3. **نظام الاستنساخ الصوتي ثنائي اللغة `FadeL Bilingual Voice Narration (XTTS v2)`** بالكامل.

In [ ]:
# === الخلية 1: دمج حزمة com.rmot.aifaceswap_8.0.xapk وتوقيعها إلى APK موحد ===
import os, zipfile, shutil, subprocess

print('1. تثبيت أدوات بناء وتوقيع الأندرويد (apktool, zipalign, apksigner, openjdk)...')
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'apktool', 'zipalign', 'apksigner', 'default-jdk-headless'], check=True)

def merge_xapk_to_universal_apk(xapk_path, output_apk='/content/Remake_AI_FaceSwap_v8.0_Universal_Signed.apk', p12_path=None, p12_pass='android'):
    work_dir = '/content/xapk_merge_work'
    shutil.rmtree(work_dir, ignore_errors=True)
    os.makedirs(work_dir, exist_ok=True)
    with zipfile.ZipFile(xapk_path, 'r') as z:
        z.extractall(work_dir)
    base_apk = os.path.join(work_dir, 'com.rmot.aifaceswap.apk')
    arm64_apk = os.path.join(work_dir, 'config.arm64_v8a.apk')
    merged_unsigned = os.path.join(work_dir, 'merged_unsigned.apk')
    shutil.copy2(base_apk, merged_unsigned)
    # Inject native arm64-v8a libraries from split APK into base APK
    if os.path.exists(arm64_apk):
        with zipfile.ZipFile(arm64_apk, 'r') as z_arm, zipfile.ZipFile(merged_unsigned, 'a') as z_base:
            for item in z_arm.infolist():
                if item.filename.startswith('lib/'):
                    z_base.writestr(item, z_arm.read(item.filename))
    aligned_apk = os.path.join(work_dir, 'merged_aligned.apk')
    subprocess.run(['zipalign', '-f', '-p', '4', merged_unsigned, aligned_apk], check=True)
    if p12_path and os.path.exists(p12_path):
        subprocess.run(['apksigner', 'sign', '--ks', p12_path, '--ks-pass', f'pass:{p12_pass}', '--out', output_apk, aligned_apk], check=True)
    else:
        debug_ks = os.path.join(work_dir, 'debug.keystore')
        subprocess.run(['keytool', '-genkey', '-v', '-keystore', debug_ks, '-storepass', 'android', '-alias', 'androiddebugkey', '-keypass', 'android', '-keyalg', 'RSA', '-keysize', '2048', '-validity', '10000', '-dname', 'CN=FadeL,O=TheNewEra,C=EG'], check=True)
        subprocess.run(['apksigner', 'sign', '--ks', debug_ks, '--ks-pass', 'pass:android', '--out', output_apk, aligned_apk], check=True)
    print('✅ تم إنتاج ملف APK الموحد والموقّع بنجاح:', output_apk)
    return output_apk

print('جاهز لدمج وتوقيع ملف XAPK وبناء تطبيق The New Era!')

# FadeL — Personal Bilingual Voice Narration (XTTS v2)

This notebook creates speech in **Arabic or English** using the voice recording you upload. It is designed for personal, non-commercial use.

## What you do
1. Run the setup cell.
2. Upload your voice recording (`MP3` or `WAV`).
3. Wait for XTTS v2 to load.
4. Enter a short test passage and generate a preview.
5. When the preview is acceptable, paste or upload a chapter and generate its MP3.

**Important:** Select a GPU runtime in Google Colab: `Runtime → Change runtime type → T4 GPU` (if available). Free GPU availability is controlled by Google and is not guaranteed.

**Language handling:** Choose `Arabic` for Arabic text and `English` for English text. For a chapter containing both languages, split it into language-consistent sections and generate each section with its matching language setting. XTTS v2 supports Arabic and English, but language selection is explicit.

## 1. Install the voice engine

This uses the maintained `coqui-tts` package. The model is downloaded by the library on first load, so the first setup can take several minutes.

In [1]:
import subprocess, sys, torch

# Colab already includes PyTorch. XTTS's current Coqui package needs torchcodec
# when PyTorch is 2.9 or newer.
torch_version = tuple(int(part) for part in torch.__version__.split("+")[0].split(".")[:2])
if torch_version >= (2, 9):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torchcodec"])

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "coqui-tts==0.28.0", "pydub"
])
print("Installation finished. If Colab asks to restart the runtime, restart it and continue from the upload cell.")

Installation finished. If Colab asks to restart the runtime, restart it and continue from the upload cell.


## 2. Upload your reference voice recording

Upload the MP3 or WAV recording you supplied in ChatGPT. The notebook converts it to a clean mono WAV reference for the model.

In [ ]:
from google.colab import files
from pathlib import Path
import subprocess, os

packaged_audio = "/content/fadel_voice_clone_source.mp3"
if os.path.exists(packaged_audio):
    audio_name = packaged_audio
    print("Using the reference recording included in the ZIP package.")
else:
    print("Select the voice recording file you uploaded in ChatGPT.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No audio file was uploaded.")
    audio_name = next(iter(uploaded.keys()))

reference_wav = "/content/reference_voice.wav"
subprocess.run([
    "ffmpeg", "-y", "-i", audio_name,
    "-ac", "1", "-ar", "24000",
    "-vn", reference_wav
], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

print("Reference audio prepared:", reference_wav)

Select the voice recording file you uploaded in ChatGPT.


KeyboardInterrupt: 

## 3. Load XTTS v2

The model supports Arabic (`ar`) and English (`en`) and can use the uploaded recording as the speaker reference.

In [ ]:
import torch
from TTS.api import TTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Execution device:", device)
if device == "cpu":
    print("WARNING: No GPU was detected. Generation may be very slow. In Colab, select Runtime → Change runtime type → GPU, then rerun the notebook.")

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("XTTS v2 is loaded.")

## 4. Generate a short voice test

Paste a short passage in the chosen language. Start with 1–3 sentences. Listen to the result before generating a full chapter.

In [ ]:
from IPython.display import Audio, display
from pydub import AudioSegment
from pathlib import Path
import re, os

def make_preview(text, language):
    text = (text or "").strip()
    if not text:
        raise ValueError("Enter a short passage first.")
    if language not in ("ar", "en"):
        raise ValueError("Language must be 'ar' or 'en'.")

    wav_path = "/content/voice_test.wav"
    mp3_path = "/content/voice_test.mp3"

    tts.tts_to_file(
        text=text,
        speaker_wav=[reference_wav],
        language=language,
        file_path=wav_path,
        split_sentences=True
    )
    AudioSegment.from_wav(wav_path).export(mp3_path, format="mp3", bitrate="192k")
    display(Audio(filename=wav_path))
    print("Preview MP3:", mp3_path)
    return mp3_path

# Edit only the passage and language values below, then run this cell.
test_language = "ar"  # use "ar" for Arabic or "en" for English
test_text = "مرحباً، هذا اختبار للصوت قبل تسجيل فصول الرواية."
# English test example:
# test_language = "en"
# test_text = "This is a short voice test before recording the novel."

preview_file = make_preview(test_text, test_language)

## 5. Generate a complete chapter

Paste the chapter text into the box below or upload a `.txt` file. The notebook splits long text into manageable sentence groups, generates each group with the same voice reference, joins the audio, and exports one MP3 file.

For best continuity, use one language per run. Arabic and English can be generated as separate sections and combined afterward if needed.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files

chapter_text_box = widgets.Textarea(
    value="",
    placeholder="Paste one chapter here...",
    description="Chapter:",
    layout=widgets.Layout(width="100%", height="260px")
)
language_box = widgets.Dropdown(
    options=[("Arabic", "ar"), ("English", "en")],
    value="ar",
    description="Language:"
)
filename_box = widgets.Text(
    value="chapter_01",
    description="File name:"
)
generate_button = widgets.Button(
    description="Generate chapter MP3",
    button_style="primary",
    icon="play"
)
chapter_output = widgets.Output()

def split_into_chunks(text, max_chars=260):
    # Keep sentence boundaries where possible; do not silently discard text.
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    sentences = re.split(r"(?<=[.!?؟۔])\s+", text)
    chunks, current = [], ""
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
        if len(sentence) > max_chars:
            if current:
                chunks.append(current)
                current = ""
            words = sentence.split()
            piece = ""
            for word in words:
                candidate = (piece + " " + word).strip()
                if len(candidate) > max_chars and piece:
                    chunks.append(piece)
                    piece = word
                else:
                    piece = candidate
            if piece:
                current = piece
            continue
        candidate = (current + " " + sentence).strip()
        if len(candidate) > max_chars and current:
            chunks.append(current)
            current = sentence
        else:
            current = candidate
    if current:
        chunks.append(current)
    return chunks

def on_generate_clicked(_):
    with chapter_output:
        clear_output()
        text = chapter_text_box.value.strip()
        lang = language_box.value
        base = re.sub(r"[^A-Za-z0-9_-]+", "_", filename_box.value.strip()).strip("_") or "chapter"
        if not text:
            print("Paste chapter text or upload a TXT file first.")
            return

        chunks = split_into_chunks(text)
        print(f"Preparing {len(chunks)} text segments. Keep this Colab tab open.")
        audio_parts = []
        for i, chunk in enumerate(chunks, start=1):
            part_path = f"/content/{base}_part_{i:04d}.wav"
            tts.tts_to_file(
                text=chunk,
                speaker_wav=[reference_wav],
                language=lang,
                file_path=part_path,
                split_sentences=True
            )
            audio_parts.append(AudioSegment.from_wav(part_path))
            print(f"Generated segment {i}/{len(chunks)}")

        combined = AudioSegment.empty()
        for part in audio_parts:
            combined += part
        out_path = f"/content/{base}.mp3"
        combined.export(out_path, format="mp3", bitrate="192k")
        print("Chapter complete:", out_path)
        display(Audio(filename=out_path))
        files.download(out_path)

generate_button.on_click(on_generate_clicked)
display(language_box, filename_box, chapter_text_box, generate_button, chapter_output)

## 6. Upload a chapter as a TXT file (optional)

If the chapter is already saved as a `.txt` file, run this cell to load it into the chapter box above.

In [ ]:
uploaded_chapter = files.upload()
for name, data in uploaded_chapter.items():
    if name.lower().endswith(".txt"):
        chapter_text_box.value = data.decode("utf-8-sig")
        print("Loaded chapter:", name)
    else:
        print("Skipped non-TXT file:", name)

## Notes and limitations

- XTTS v2 is a pretrained voice-cloning model; this workflow does **not** train a new model from scratch.
- The recording you uploaded is the speaker reference. Clear, consistent speech generally gives better results than noisy or heavily processed audio.
- Arabic and English are supported model languages. Actual accent fidelity, pronunciation, and voice similarity must be judged from the generated preview; they are not guaranteed.
- Keep the reference recording and generated audio private. Do not publish a public Colab/Gradio sharing link containing personal voice data.
- Google Colab free GPU access is subject to Google's availability limits.
- XTTS v2 is distributed under the Coqui Public Model License. This notebook is intended for the personal, non-commercial use you selected. Review the license before any other use.

## 🚀 القسم الإضافي (v12 ULTIMATE): تشغيل محرك `linoyts/Flux2-Klein-Face-Swap` (`app.py`) ومحرك `inswapper_128.onnx + GFPGANv1.4` (`predict.py` / `cog.yaml`) على GPU مجاني
تم دمج ملفاتك المرفوعة (`app.py`, `index.html`, `start.bat`, `predict.py`, `cog.yaml`, `DIRECTOR.txt`) بالكامل هنا لتشغيلها بنقرة واحدة:

In [ ]:
# تثبيت وتشغيل خادم Face Swap Studio (app.py + linoyts/Flux2-Klein-Face-Swap) مع رابط عام للهاتف
!pip install -q flask gradio_client pycloudflared
import os
os.makedirs('templates', exist_ok=True)
os.makedirs('uploads', exist_ok=True)
os.makedirs('results', exist_ok=True)
APP_PY = "from flask import Flask, request, jsonify, send_from_directory, render_template\nfrom gradio_client import Client, handle_file\nimport os\nimport uuid\nimport shutil\n\napp = Flask(__name__)\n\nUPLOAD_FOLDER = \"uploads\"\nRESULT_FOLDER = \"results\"\nos.makedirs(UPLOAD_FOLDER, exist_ok=True)\nos.makedirs(RESULT_FOLDER, exist_ok=True)\n\nprint(\"Connecting to Hugging Face Space...\")\nclient = Client(\"linoyts/Flux2-Klein-Face-Swap\")\nprint(\"Connected!\")\n\n@app.route(\"/\")\ndef index():\n    return render_template(\"index.html\")\n\n@app.route(\"/api/swap\", methods=[\"POST\"])\ndef swap():\n    try:\n        ref_file = request.files.get(\"reference_face\")\n        tgt_file = request.files.get(\"target_image\")\n\n        if not ref_file or not tgt_file:\n            return jsonify({\"error\": \"You must upload both images\"}), 400\n\n        ref_id = str(uuid.uuid4())\n        tgt_id = str(uuid.uuid4())\n\n        ref_path = os.path.join(UPLOAD_FOLDER, ref_id + \".jpg\")\n        tgt_path = os.path.join(UPLOAD_FOLDER, tgt_id + \".jpg\")\n\n        ref_file.save(ref_path)\n        tgt_file.save(tgt_path)\n\n        print(\"Received images: \" + ref_path + \" , \" + tgt_path)\n\n        width, height = client.predict(\n            target_image=handle_file(tgt_path),\n            api_name=\"/update_dimensions_from_image\"\n        )\n        print(\"Dimensions: \" + str(width) + \"x\" + str(height))\n\n        result = client.predict(\n            reference_face=handle_file(ref_path),\n            target_image=handle_file(tgt_path),\n            seed=0,\n            randomize_seed=True,\n            width=width,\n            height=height,\n            num_inference_steps=4,\n            guidance_scale=1,\n            api_name=\"/face_swap\"\n        )\n\n        before_path = result[0][0]\n        after_path = result[0][1]\n        seed_used = result[1]\n\n        print(\"Done! Seed: \" + str(seed_used))\n\n        result_id = str(uuid.uuid4())\n        result_filename = result_id + \".webp\"\n        result_dest = os.path.join(RESULT_FOLDER, result_filename)\n        shutil.copy(after_path, result_dest)\n\n        return jsonify({\n            \"success\": True,\n            \"result_url\": \"/results/\" + result_filename,\n            \"seed\": seed_used,\n            \"width\": width,\n            \"height\": height\n        })\n\n    except Exception as e:\n        print(\"Error: \" + str(e))\n        return jsonify({\"error\": str(e)}), 500\n\n@app.route(\"/results/<filename>\")\ndef get_result(filename):\n    return send_from_directory(RESULT_FOLDER, filename)\n\nif __name__ == \"__main__\":\n    print(\"Server running at http://localhost:5000\")\n    app.run(host=\"0.0.0.0\", port=5000, debug=False)"
with open('app.py', 'w', encoding='utf-8') as f:
    f.write(APP_PY)
print('✅ app.py ready (Connected to linoyts/Flux2-Klein-Face-Swap)')


In [ ]:
# تثبيت بيئة محرك GPU الفائق (predict.py + cog.yaml: insightface==0.7.3 + inswapper_128.onnx + GFPGANv1.4.pth + realesrgan)
!pip install -q insightface==0.7.3 onnx==1.14.0 onnxruntime-gpu gfpgan==1.3.8 realesrgan opennsfw2==0.10.2 mediapipe
COG_YAML = "build:\n  gpu: true\n  cuda: \"11.8\"\n  python_version: \"3.9\"\n  python_packages:\n    - \"opencv-contrib-python==4.7.0.72\"\n    - \"ftfy==6.1.1\"\n    - \"scipy==1.9.3\"\n    - \"Pillow==9.4.0\"\n    - \"mediapipe\"\n    - \"numpy==1.24.3\"\n    - \"opencv-python==4.7.0.72\"\n    - \"onnx==1.14.0\"\n    - \"insightface==0.7.3\"\n    - \"psutil==5.9.5\"\n    - \"tensorflow>=2.0.0\"\n    - \"opennsfw2==0.10.2\"\n    - \"gfpgan==1.3.8\"\n    - \"realesrgan\"\n    - \"torchvision==0.12.0\"\n    - \"onnxruntime-gpu==1.15.0\"\n  system_packages:\n    - ffmpeg\n    - libsm6\n    - libxext6\npredict: \"predict.py:Predictor\"\n"
PREDICT_PY = "# Prediction interface for Cog \u2699\ufe0f\n# For more information, see: https://github.com/replicate/cog/blob/main/docs/python.md\nimport uuid\n\n# Import necessary classes from the cog library for creating a predictor interface.\nfrom cog import BasePredictor, Input\nfrom cog import Path as CogPath\nimport sys\nimport time\nimport torch\n# Import custom modules for global settings and model management.\nimport core.globals\nimport core.model_manager\n# Import the pre-configured face enhancer and face swapping modules.\nfrom core.face_enhancer import face_enhancer\nfrom core.face_swap_picture import faceSwapPicture\n\n# Check if a CUDA-enabled GPU is available.\nif not torch.cuda.is_available():\n    # If not, set the execution provider to CPU for ONNX Runtime.\n    core.globals.providers = ['CPUExecutionProvider']\n    print(\"No GPU detected. Using CPU instead.\")\n\nimport os\n# Import OpenCV for image processing.\nimport cv2\n# Import Iterator for type hinting the return value of the predict method.\nfrom typing import Iterator\n# Import subprocess functions to run shell commands.\nfrom subprocess import call, check_call\n\n# Import utility functions for image checks and modifications.\nfrom core.utils import add_white_side\n# Import a utility function to detect faces in an image.\nfrom core.config import get_face\n\n\ndef status(string):\n    \"\"\"Prints a status message to the console.\"\"\"\n    print(\"Status: \" + string)\n\n\ndef run_cmd(command):\n    \"\"\"Executes a shell command.\"\"\"\n    try:\n        call(command, shell=True)\n    except KeyboardInterrupt:\n        print(\"Process interrupted\")\n        sys.exit(1)\n\n\nclass Predictor(BasePredictor):\n    \"\"\"\n    A predictor class that defines the model's setup and prediction logic.\n    Inherits from cog.BasePredictor.\n    \"\"\"\n    def setup(self):\n        \"\"\"\n        Initial setup method. This runs once when the model is loaded into memory.\n        \"\"\"\n        # A delay to allow system resources to stabilize.\n        time.sleep(10)\n        # Run nvidia-smi to verify GPU status.\n        check_call(\"nvidia-smi\", shell=True)\n        # Assert that a CUDA-enabled GPU is available, otherwise the program will stop.\n        assert torch.cuda.is_available()\n\n    def predict(\n            self,\n            source: CogPath = Input(description=\"Source image with the face to be used.\"),\n            target: CogPath = Input(description=\"Target image where the face will be swapped.\"),\n            is_enhancer: bool = Input(description=\"Enable face enhancement on the result.\", default=False),\n            keep_fps: bool = Input(description=\"Keep FPS (not used in this image-to-image context).\", default=True),\n            keep_frames: bool = Input(description=\"Keep Frames (not used in this image-to-image context).\", default=True),\n    ) -> Iterator[CogPath]:\n        \"\"\"\n        The main prediction method. This runs every time a new prediction is requested.\n        \"\"\"\n        # Print the input paths and options for debugging purposes.\n        print(\"source: \", source)\n        print(\"target: \", target)\n        print(\"keep_fps: \", keep_fps)\n        print(\"keep_frames: \", keep_frames)\n\n        # Convert the CogPath objects to standard string paths.\n        source = str(source)\n        target = str(target)\n\n        # Add a white border to the source image to help with face detection at the edges.\n        source = add_white_side(source)\n\n        # --- Validation Step ---\n        # Read the source image and check if a face is present.\n        source_face = get_face(cv2.imread(source))\n        if not source_face:\n            print(\"\\n[WARNING] No face detected in source image. Please try with another one.\\n\")\n            return\n\n        # Read the target image and check if a face is present.\n        target_face = get_face(cv2.imread(target))\n        if not target_face:\n            print(\"\\n[WARNING] No face detected in target image. Please try with another one.\\n\")\n            return\n\n        # --- Face Swapping Step ---\n        # Perform the face swap from the source image to the target image.\n        output = faceSwapPicture.swap(source=source, target=target)\n\n        # Conditionally enhance the face in the output image if the option is enabled.\n        if is_enhancer:\n            result = face_enhancer.enhancer_image(target_path=output, output_path=output)\n\n        # Yield the path to the final output image.\n        yield CogPath(output)\n        # Print a final status message.\n        status(\"img swap successful!\")\n        return"
with open('cog.yaml', 'w', encoding='utf-8') as f: f.write(COG_YAML)
with open('predict.py', 'w', encoding='utf-8') as f: f.write(PREDICT_PY)
print('✅ predict.py & cog.yaml (inswapper_128 + GFPGANv1.4) written to Colab workspace!')
